# DL

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        # 은닉층(Hidden Layer)의 크기와 개수를 컴퓨터가 조합해 봅니다.
        'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(64, 32), (128, 64, 32), (64,)]),
        # 과적합 방지를 위한 L2 규제 (XGB의 reg_lambda와 비슷)
        'alpha': trial.suggest_float('alpha', 0.0001, 0.1, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 0.001, 0.05, log=True),
        'max_iter': 1000,          # 충분한 반복 횟수
        'early_stopping': True,    # 신경망 자체적인 조기 종료 켜기
        'random_state': 42
    }
    
    model_opt = MLPClassifier(**params)
    
    # 🚨 신경망도 내부적으로 분할하므로 여기서 eval_set은 넣지 않습니다.
    model_opt.fit(X_opt_tr, y_opt_tr)
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

# 🌟 Optuna 결과 덮어쓰기
    model = MLPClassifier(
        **study.best_params,
        max_iter=1000,
        early_stopping=True,
        random_state=42 + fold
    )
    
    # 🚨 eval_set 없음
    model.fit(X_tr, y_tr)
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...
✅ 병합 완료! (전체 학습 데이터 X: (200000, 44), y: (200000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


c:\Users\goiji\.conda\envs\goji60000-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-09 13:05:08,850] A new study created in memory with name: no-name-f6fda3dd-1f9b-4328-b34b-2cb970f28710
c:\Users\goiji\.conda\envs\goji60000-env\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (64, 32) which is of type tuple.
  optuna_warn(message)
c:\Users\goiji\.conda\envs\goji60000-env\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (128, 64, 32) which is of type tuple.
  optuna_warn(message)
c:\Users\goiji\.conda\envs\goji60000-env\Lib\site

🎉 탐색 완료! 최고 Validation AUC: 0.5918

🚀 XGBoost K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5896)
   - Fold 2 완료 (ROC-AUC: 0.5884)
   - Fold 3 완료 (ROC-AUC: 0.5893)
   - Fold 4 완료 (ROC-AUC: 0.5818)
   - Fold 5 완료 (ROC-AUC: 0.5867)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.5654
F1 점수 (F1 Score): 0.4759
AUC (ROC-AUC):  0.5866
Fold별 AUC: [0.5896, 0.5884, 0.5893, 0.5818, 0.5867]


## Main

In [7]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

print("📥 PyTorch 딥러닝을 위한 스케일링 데이터를 불러옵니다...")

# 🚨 딥러닝은 반드시 스케일링된 데이터를 사용해야 합니다.
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')
y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

# 테스트 데이터를 미리 텐서(딥러닝 전용 데이터 타입)로 변환해 둡니다.
X_test_tensor = torch.FloatTensor(X_test.values)

# ==========================================
# 🧩 부품 1: 신경망 아키텍처 (도면 설계)
# ==========================================
class ReturnRiskNet(nn.Module):
    def __init__(self, input_dim):
        super(ReturnRiskNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),   # 학습을 안정적으로 만들어주는 부품
            nn.ReLU(),
            nn.Dropout(0.3),      # 과적합 방지
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(32, 1),
            nn.Sigmoid()          # 0~1 사이 확률값 출력
        )

    def forward(self, x):
        return self.network(x)

# ==========================================
# 2. K-Fold 설정 및 PyTorch 학습 루프 시작
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_scores = []

# 학습용 파라미터 (원래라면 Optuna로 찾았을 값들을 고정값으로 설정)
EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 0.005

print("\n🚀 PyTorch K-Fold 딥러닝 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold} ---")
    
    # 1) Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx].values, X.iloc[val_idx].values
    y_tr, y_va = y.iloc[train_idx].values, y.iloc[val_idx].values
    
    # 2) 🧩 부품 2: 데이터 로더 (텐서로 변환하고 식판에 담기)
    # y값의 형태를 (N,) 에서 (N, 1) 로 세워주어야 딥러닝이 인식합니다.
    train_dataset = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr).unsqueeze(1))
    val_dataset = TensorDataset(torch.FloatTensor(X_va), torch.FloatTensor(y_va).unsqueeze(1))
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 3) 모델, 손실 함수, 최적화 도구 세팅
    model = ReturnRiskNet(input_dim=X.shape[1])
    criterion = nn.BCELoss() # 이진 분류용 손실 함수
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 4) 🧩 부품 3: 학습 루프 (수동 변속 과정)
    for epoch in range(EPOCHS):
        model.train() # 학습 모드 ON
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()            # 1. 기울기 초기화
            predictions = model(batch_X)     # 2. 예측
            loss = criterion(predictions, batch_y) # 3. 오차 계산
            loss.backward()                  # 4. 역전파 (어디 고칠지 계산)
            optimizer.step()                 # 5. 가중치 업데이트
            
    # 5) 검증 데이터 예측 및 OOF 저장
    model.eval() # 평가 모드 ON (Dropout 등 정지)
    with torch.no_grad(): # 예측 시에는 기울기 계산을 끔(메모리 절약)
        val_proba = model(torch.FloatTensor(X_va)).squeeze().numpy()
        oof_pred[val_idx] = val_proba
        
        # 테스트 데이터 예측 (5로 나누어 누적)
        test_proba = model(X_test_tensor).squeeze().numpy()
        test_pred += test_proba / skf.n_splits
        
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
oof_label = (oof_pred >= 0.5).astype(int)
print(f"\n🏆 최종 PyTorch OOF ROC-AUC: {overall_auc:.4f}")



📥 PyTorch 딥러닝을 위한 스케일링 데이터를 불러옵니다...

🚀 PyTorch K-Fold 딥러닝 학습을 시작합니다...

--- Fold 1 ---


KeyboardInterrupt: 

In [5]:
# ==========================================
# 4. 대시보드 연동용 CSV 일괄 추출 (PyTorch용)
# ==========================================
model_name = "PyTorch" 
print("\n💾 대시보드 및 제출용 CSV 파일 추출을 시작합니다...")

# 1. Validation 예측 결과
val_df = pd.DataFrame({'pred_prob': oof_pred, 'pred_label': (oof_pred >= 0.5).astype(int), 'returned': y.values})
val_df.to_csv('val_predictions.csv', index=False)

# 2. Test 예측 결과 및 제출용
test_df = pd.DataFrame({'order_id': test_id_df['order_id'], 'pred_prob': test_pred, 'pred_label': (test_pred >= 0.5).astype(int)})
test_df.to_csv('test_predictions.csv', index=False)
test_df[['order_id', 'pred_prob']].rename(columns={'pred_prob': 'returned'}).to_csv(f'{model_name}_submission.csv', index=False)

print("✅ [1/2] 예측 결과 및 제출용 파일 생성 완료!")
print("⚠️ [2/2] PyTorch 신경망은 자체적인 변수 중요도를 제공하지 않아 생략합니다.")

X.to_csv('X_val_unscaled.csv', index=False)
pd.DataFrame({'returned': y}).to_csv('y_val.csv', index=False)
print("\n🎉 모든 파이토치 모델 학습 및 추출이 완료되었습니다!")


💾 대시보드 및 제출용 CSV 파일 추출을 시작합니다...
✅ [1/2] 예측 결과 및 제출용 파일 생성 완료!
⚠️ [2/2] PyTorch 신경망은 자체적인 변수 중요도를 제공하지 않아 생략합니다.

🎉 모든 파이토치 모델 학습 및 추출이 완료되었습니다!
